# ST 554 Analysis of BigData - Fianl Project
* Name: Yujin Kim
* Course: ST 554 (601) Spring 2026 Analysis of Big Data
* Assignment: Final Project

# Introduction
This project focuses on building a machine learning pipeline using PySpark to model power consumption. The goal is to predict the Power_Zone_3 variable using other environmental and operational variables.

The project consists of two main components:

(1) regression model is developed using an Elastic Net approach with cross-validation to identify the optimal model parameters. 

(2) streaming pipeline is implemented to simulate real-time data processing, where incoming data is continuously processed and predictions are generated.

The streaming system is designed to read data from a folder, apply the trained model, and output predictions along with residual values in real time.

In [63]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = SparkSession.builder.appName("FinalProject").getOrCreate()

# 1. Fitting Model
In this step, the dataset is loaded using pandas and then converted into a Spark DataFrame. 
This is required because the project instructions specify using `pd.read_csv()` before transitioning to Spark.

The dataset contains power consumption data along with environmental variables such as temperature, humidity, and wind speed. The target variable for prediction is Power_Zone_3.

## 1.1. Data Loading

In [64]:
#Instruction: You should read this data into a standard pandas data frame using the pd.read_csv() function.
df_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [65]:
#Instruction: Convert this to a spark data frame
spark_df = spark.createDataFrame(df_pd)
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [66]:
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## 1.2. Data Transformation Pipeline

A series of transformations are applied to prepare the data for modeling. These transformations are implemented using PySpark MLlib and combined into a pipeline.

- The Hour variable is cast to a numeric type.
- The Hour variable is binarized to represent day and night.
- The Month variable is converted into a one-hot encoded format.
- PCA is applied to reduce dimensionality of environmental variables.
- All features are assembled into a single feature vector.


Instruction:
We are going to treat the Power_Zone_3 variable as our response variable.
We can use all of the other variables as predictors. (Imagine we know that the Power_Zone_3 reading
is going to go offline in the future and we need to be able to predict that value appropriately.)

In [67]:
#cast hour and rename response variable
sql_transformer = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double, Power_Zone_3 AS label
    FROM __THIS__""")
#response variable Power_Zone_3 is copied and renamed as "label".

Instruction: The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the
variable as a DoubleType.
Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

In [68]:
#create new binary variable
hour_binarizer = Binarizer(threshold = 6.5, inputCol = "Hour_double", outputCol = "Hour_binary")

Instruction: One-hot encode the Month column

In [69]:
month_encoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_encoded"])

Instruction: Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and
Diffuse_Flows columns.

In [70]:
#Instruction: Use a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol = "pca_input")

In [71]:
#Instruction: We’ll use two PCs in our transformation.
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features") #define PCA transformer

Instruction: Use VectorAssembler() to put your predictors into a features. Use the
* two fitted PCA features
* binary Hour variable
* Power_Zone_1
* Power_Zone_2
* Month indicator variables

In [72]:
feature_assembler = VectorAssembler(
    inputCols = ["pca_features", "Hour_binary", "Power_Zone_1", "Power_Zone_2", "Month_encoded"],
    outputCol = "features")

Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.

In [73]:
lr = LinearRegression(featuresCol = "features", labelCol = "label", predictionCol = "prediction")

Instruction: The transformations below should each use an MLlib function that can be put into a pipeline

In [74]:
#Build full pipeline
pipeline = Pipeline(stages = [sql_transformer, hour_binarizer, month_encoder, pca_assembler, pca, feature_assembler, lr])

## 1.3. Model Training (Elastic Net + CV)
An Elastic Net regression model is trained using PySpark's LinearRegression. 
Cross-validation is applied to determine the optimal combination of hyperparameters.
The model is evaluated using 5-fold cross-validation, with RMSE as the performance metric.


Instruction: Now you’ll then use the CrossValidator() function and the LinearRegression() function to fit an
elastic net model.

In [75]:
#define evaluator of model performance
evaluator = RegressionEvaluator(labelCol = "label", predictionCol = "prediction", metricName = "rmse")

Instruction: You should do the following grid for the regParam and elasticNetParam: All combinations of
* regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
* elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

In [76]:
#define parameter grid
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = (ParamGridBuilder().addGrid(lr.regParam, param_values).addGrid(lr.elasticNetParam, param_values).build())

Instruction: Now fit the model using 5-fold CV with rmse as your criterion!

In [77]:
#set up 5-fold cross validation
cv = CrossValidator(estimator = pipeline,
                    estimatorParamMaps = param_grid,
                    evaluator = evaluator,
                    numFolds = 5,
                    seed = 123)

In [78]:
#fit the model
cv_model = cv.fit(spark_df)

26/04/29 14:44:35 WARN Instrumentation: [f1571adc] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:44:35 WARN Instrumentation: [f1571adc] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:44:36 WARN Instrumentation: [406d6f53] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:44:36 WARN Instrumentation: [406d6f53] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:44:37 WARN Instrumentation: [a23d5c21] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:44:38 WARN Instrumentation: [a23d5c21] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:44:38 WARN Instrumentation: [1c81dd93] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:44:39 WARN Instrumentation: [1c81dd93] Cholesky solv

Instruction: Report the optimal values chosen for the tuning parameters

In [79]:
#report optimal tuning parameters
best_model = cv_model.bestModel
best_lr_model = best_model.stages[-1]

best_reg_param = best_lr_model.getRegParam()
best_elastic_net_param = best_lr_model.getElasticNetParam()

print("Best regParam:", best_reg_param)
print("Best elasticNetParam:", best_elastic_net_param)

Best regParam: 0.05
Best elasticNetParam: 0.1


Instruction: Report the CV error

In [80]:
# cv_model.avgMetrics stores the average CV RMSE for each parameter combination. The smallest value corresponds to the optimal CV error.
cv_errors = cv_model.avgMetrics
best_cv_error = min(cv_errors)

print("Best CV RMSE:", best_cv_error)

Best CV RMSE: 2147.8113505767924


## 1.4. Model Evaluation
After training, the model is evaluated on the training dataset. 
Predictions are generated and compared with actual values to compute RMSE.


Instruction: Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and
evaluating on the entire training set

In [81]:
# Apply the final selected model to the training data
training_predictions = cv_model.transform(spark_df)
training_rmse = evaluator.evaluate(training_predictions)

print("Training RMSE:", training_rmse)

Training RMSE: 2147.0973169293934


Instruction: Take the outputted transformations from the model (the predictions) and create a residual column
(label - prediction). The .withColumn() method is handy here. Print out a data frame with these
residuals, the label column, and the predictions

In [82]:
# Create a residual column from the prediction results
training_predictions_with_residuals = training_predictions.withColumn("residual", F.col("label") - F.col("prediction"))

training_predictions_with_residuals.select("label", "prediction", "residual").show(20)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20878.850660788543| -637.886800788543|
|20131.08434| 18660.22726654401|1470.8570734559908|
|19668.43373| 18204.75215311454|1463.6815768854613|
|18899.27711|17590.648498339142|1308.6286116608571|
|18442.40964|16997.301986456878| 1445.107653543124|
|18130.12048|16517.686723494306|1612.4337565056958|
|17945.06024|16093.246141053509|  1851.81409894649|
|17459.27711|15722.695360253945| 1736.581749746054|
|17025.54217|15271.043828662865|1754.4983413371356|
|16794.21687|14938.348764665692| 1855.868105334308|
|16638.07229|14652.383721423883|1985.6885685761172|
|16395.18072|14414.900662730442|1980.2800572695578|
|16117.59036|14082.889275686905|2034.7010843130956|
| 15822.6506| 13624.88209143617|2197.7685085638313|
|15672.28916|13450.340775468403| 2221.948384531597|
|15597.10843| 13302.24639456202|  2294.86203543798|
|15510.36145

# 2. Streaming Part
There is another file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv

Download this file and store it where your .py file you’ll create can find it. We’ll be randomly sampling rows from this to output to .csv files that you’ll be reading in.

In [83]:
import pandas as pd
import os

In [84]:
#read sreaming data
stream_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv")

In [85]:
#Save original file for streaming
stream_pd.to_csv("power_streaming_data.csv", index = False)
stream_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,4.805,76.2,0.081,0.059,0.134,20421.26582,12908.20669,14590.84337,1,3
1,4.212,78.3,0.081,0.117,0.082,21393.41772,13575.68389,14862.65060,1,5
2,4.304,76.0,0.082,0.048,0.152,19983.79747,12342.85714,13492.04819,1,7
3,4.489,74.3,0.082,0.081,0.119,18167.08861,11551.36778,11600.96386,1,7
4,4.509,74.5,0.084,6.643,6.494,19837.97468,11945.28875,11178.79518,1,8


## 2.1. Reading a Stream
A streaming DataFrame is created by monitoring a folder where CSV files are continuously added. 
The schema is predefined to ensure consistency with the training data.


Instruction: We’re going to read in a stream in the form of .csv files. Create a folder where you will be sending
your .csv files.

In [86]:
#generate folder for Streaming file
stream_folder = "stream_folder"
os.makedirs(stream_folder, exist_ok = True)

Instruction: Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)

In [87]:
stream_schema = spark_df.schema

Instruction: Set up the readStream. Be sure to add header = True as you’ll likely be outputting files with a
header and we don’t need to read that in.

In [88]:
stream_df = spark.readStream.option("header", True).schema(stream_schema).csv(stream_folder)

## 2.2. Transform / Aggregation Step
The trained model is applied to incoming streaming data to generate predictions. A residual column is also calculated.


Instruction: Now, we’ll do two separate things on the stream and join them together:

Instruction: With your stream, use your model transformer to obtain predictions from the incoming data. On
the resulting predictions also create a residual column as noted in the previous section (return
only the label, prediction and residual columns from this part)

In [89]:
#Apply the trained model to the stream
stream_predictions = cv_model.transform(stream_df)

In [90]:
#create residual column
stream_predictions_residual = stream_predictions.withColumn("residual", F.col("label") - F.col("prediction"))

In [91]:
#Select only the required columns from the prediction results
stream_prediction_output = stream_predictions_residual.select("label", "prediction", "residual")

Instruction: We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called label.

In [92]:
#Create a label column from the original stream
stream_label = stream_df.withColumnRenamed("Power_Zone_3", "label")

Instruction: Now join your above transform with this stream based on the label variable which should be common to both!

In [93]:
#Join the results of the two stream transformations
joined_stream = stream_prediction_output.join(stream_label, on = "label")

## 2.3. Writing Step
The results are written to the console using append mode. The streaming query runs continuously and updates as new data arrives.

Instruction: Now write your stream to the console using the append output mode.

In [112]:
query = joined_stream.writeStream.outputMode("append").format("console").start()

26/04/29 15:14:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-93a9a406-f303-404a-8497-56a3831b9199. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 15:14:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14833.73494|16612.509269396956|-1778.774329396956|      13.07|   47.75|     4.921|                370.3|        382.3| 32178.22785| 21082.06687|    1|  10|
|15832.64556|18701.019006447743|-2868.373446447742|      21.57|    86.9|     0.277|                0.066|        0.141| 38548.67257| 25682.74428|    9|  22|
|11178.75383| 16298.40294417972| -5119.64911417972|      22.77|   66.76|     4.917|                609.3|         89.3

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|22237.09091|20350.128910895946|1886.9619991040527|       16.4|    73.6|     0.081|                0.033|        0.126| 33772.57266| 18663.54379|    4|  23|
|10457.87234| 11321.22982307218|-863.3574830721809|      18.51|    90.3|     0.066|                31.55|        26.35| 30262.05689| 20236.92946|   10|   8|
|14260.64516| 13921.44112310072|339.20403689928025|        9.8|    87.9|     0.077|                0.044|        0.148

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16170.96774| 15621.30868919299| 549.6590508070094|      17.37|    80.2|     0.075|                0.044|        0.126| 26373.44681| 15398.78049|    3|   0|
|25612.25806|25998.167503346493|-385.9094433464925|      15.72|    80.4|     4.913|                0.029|        0.141| 44492.93617| 26191.46341|    3|  20|
|14932.04819|15592.446747920289| -660.398557920289|      6.539|    82.8|     0.087|                0.073|        0.137

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14707.74194|14403.879450149909|303.86248985009115|      11.85|    86.5|     0.083|                0.055|        0.145| 24486.12766| 14703.65854|    3|   5|
|13485.10725|12211.077036524324|1274.0302134756748|      21.12|    73.1|      4.92|                0.066|        0.115| 26614.51327| 16207.48441|    9|   4|
|27072.60188|25726.978070848905|1345.6238091510932|      28.67|   68.46|     4.906|                685.0|        309.0

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 16106.0241|14738.266242867014| 1367.7578571329868|      10.84|    78.4|     0.073|                0.062|        0.156| 23914.93671|  16132.5228|    1|   0|
| 22308.3871| 21173.35663028281| 1135.0304697171887|      11.27|    80.3|     0.075|                0.048|        0.189| 37292.93617| 21878.04878|    3|  22|
|10501.14573|10489.951535043643| 11.194194956357023|      11.39|   65.58|     4.916|                0.059|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16809.31563|15237.493981502948| 1571.8216484970526|      22.71|    84.7|     0.279|                0.073|        0.119| 31119.29204| 18972.97297|    9|   0|
|24670.84337|23199.509642099547| 1471.3337279004518|      16.01|   55.27|     4.922|                0.066|        0.119| 39584.81013| 22924.01216|    1|  19|
|16168.00817|13966.092819916636| 2201.9153500833636|      25.53|   61.79|     0.309|                613.7|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15863.22581|17560.464783144576|-1697.2389731445764|      16.63|   57.33|     0.083|                309.5|        285.8| 33934.97872| 21095.12195|    3|  15|
|27433.73041|25680.692896958084|  1753.037513041916|       27.6|   68.69|     4.921|                821.0|         82.2| 40767.50277| 26800.42239|    8|  13|
|9726.770708|11438.019094898513|-1711.2483868985128|      16.83|   58.28|     0.087|                455.5|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|27391.59875| 26923.95746052033|  467.6412894796704|      29.76|   45.28|     4.908|                831.0|        38.63| 42263.44062| 28990.07392|    8|  14|
|12751.36778|10137.671287130483|  2613.696492869518|      17.83|    87.8|     4.921|                0.062|        0.141| 26165.77681| 14620.33195|   10|   5|
|17780.36364|18997.703834168104| -1217.340194168104|      16.33|    78.7|     0.071|                170.6|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12372.03647|14182.977445071983|-1810.9409750719842|      21.43|    71.6|     4.925|                426.5|        267.7| 36979.95624| 23470.95436|   10|  12|
|16145.36683|16573.643476826823| -428.2766468268219|      12.47|   40.53|     0.088|                212.6|        231.3|  31557.9661| 18900.91185|    2|  17|
|16145.36683|16573.643476826823| -428.2766468268219|      12.47|   40.53|     0.088|                212.6|      

# 3. Produce Data
A separate Python script is used to simulate streaming data. 
The script randomly samples rows from the original dataset and writes them as CSV files into the streaming folder at fixed intervals.


Instruction: You should have the file we’ll use for streaming data downloaded and in a place you can locate. Create a .py file that reads that into a pandas (regular) data frame and does the following.

In [113]:
for q in spark.streams.active:
    q.stop()

26/04/29 15:18:25 WARN DAGScheduler: Failed to cancel job group 60af8706-3ea5-4fc3-a94f-723b7ca47d58. Cannot find active jobs for it.
26/04/29 15:18:26 WARN DAGScheduler: Failed to cancel job group 60af8706-3ea5-4fc3-a94f-723b7ca47d58. Cannot find active jobs for it.


In [114]:
spark.streams.active

[]

# Conclusion
In this project, a complete machine learning and streaming pipeline was developed using PySpark. 
The Elastic Net regression model demonstrated the ability to predict power consumption effectively based on environmental variables.

The streaming component simulated real-time data processing, showing how a trained model can be applied dynamically to incoming data. 
The project provides practical experience with distributed data processing, machine learning pipelines, and streaming systems using Spark.

In [111]:
import glob

for f in glob.glob("stream_folder/*.csv"):
    os.remove(f)